# Churn v2: MLP pequena y presupuesto secuencial

Segunda iteracion del Hackathon 3. El objetivo es ordenar y seleccionar intervenciones para maximizar el profit con presupuesto inicial de **S/1 000**, costo de **S/10** y recompensa de **S/100** por churn correctamente intervenido.

El split externo es 80/20 estratificado con semilla 42. El modelo oficial se selecciona solo con predicciones out-of-fold de train; test se abre una unica vez. La tabla retrospectiva en test se presenta solo como diagnostico y no como seleccion valida.

## TL;DR

Esta celda se completa conceptualmente con los artefactos finales en `/kaggle/working`: `second_iteration.md`, metricas, ledger y curvas. El notebook no promete mejora antes de ejecutar: conserva y reporta el resultado real.

## 1. Configuracion y GPU

Se fija semilla 42, se exige una NVIDIA Tesla T4 y se registran versiones. PyTorch, CatBoost, LightGBM y XGBoost usan GPU; Logistic Regression y Random Forest quedan como baselines CPU.

In [ ]:
import copy
import json
import random
import subprocess
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

SEED = 42
INITIAL_BUDGET = 1000
INTERVENTION_COST = 10
SUCCESS_REWARD = 100
BREAK_EVEN_PROBABILITY = INTERVENTION_COST / SUCCESS_REWARD
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert torch.cuda.is_available(), "CUDA no esta disponible"
assert "T4" in gpu_name, f"Se esperaba NVIDIA Tesla T4; se obtuvo {gpu_name!r}"
DEVICE = torch.device("cuda:0")

run_summary = {
    "status": "running",
    "stage": "setup",
    "seed": SEED,
    "gpu": gpu_name,
    "cuda_available": True,
    "artifacts": [],
}

def save_summary():
    (OUTPUT_DIR / "run_summary.json").write_text(
        json.dumps(run_summary, indent=2, ensure_ascii=False), encoding="utf-8"
    )

save_summary()
print({"gpu": gpu_name, "torch_cuda": torch.cuda.is_available(), "device": str(DEVICE)})

## 2. Datos y split

`TotalCharges` vacio corresponde a `tenure == 0` y se convierte a cero. `customerID` no entra al modelo, pero se conserva para trazabilidad y desempate determinista.

In [ ]:
dataset_candidates = list(Path("/kaggle/input").rglob("telco.csv"))
assert dataset_candidates, "No se encontro telco.csv en /kaggle/input"
DATA_PATH = dataset_candidates[0]

df = pd.read_csv(DATA_PATH)
required_columns = {"customerID", "TotalCharges", "tenure", "Churn"}
assert required_columns.issubset(df.columns), required_columns - set(df.columns)
assert len(df) == 7043, f"Se esperaban 7043 filas; se obtuvieron {len(df)}"
assert df["customerID"].is_unique, "customerID debe ser unico"

total_charges = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")
assert (total_charges.isna() == (df["tenure"] == 0)).all()
df["TotalCharges"] = total_charges.fillna(0.0)
df["ChurnFlag"] = (df["Churn"] == "Yes").astype(np.int8)

feature_columns = [c for c in df.columns if c not in {"customerID", "Churn", "ChurnFlag"}]
numeric_columns = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_columns = [c for c in feature_columns if c not in numeric_columns]

all_indices = np.arange(len(df))
train_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.20,
    stratify=df["ChurnFlag"],
    random_state=SEED,
)
assert (len(train_indices), len(test_indices)) == (5634, 1409)
assert set(train_indices).isdisjoint(test_indices)

X_train = df.loc[train_indices, feature_columns].reset_index(drop=True)
y_train = df.loc[train_indices, "ChurnFlag"].to_numpy()
ids_train = df.loc[train_indices, "customerID"].astype(str).to_numpy()
X_test = df.loc[test_indices, feature_columns].reset_index(drop=True)
y_test = df.loc[test_indices, "ChurnFlag"].to_numpy()
ids_test = df.loc[test_indices, "customerID"].astype(str).to_numpy()
assert abs(y_train.mean() - y_test.mean()) < 0.002

run_summary.update({
    "stage": "data_ready",
    "dataset_path": str(DATA_PATH),
    "rows": len(df),
    "train_rows": len(X_train),
    "test_rows": len(X_test),
})
save_summary()
print({"train": X_train.shape, "test": X_test.shape, "churn_train": y_train.mean(), "churn_test": y_test.mean()})

## 3. Preprocesamiento, MLP y simulador

La MLP implementa `inputs -> 64 ReLU -> Dropout(0.2) -> 32 ReLU -> Dropout(0.1) -> 1`. Se entrena con logits por estabilidad numerica y aplica sigmoid al producir probabilidades.

In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_columns),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_columns),
    ], verbose_feature_names_out=False)


class ChurnMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 1),
        )

    def forward(self, features):
        return self.network(features).squeeze(1)


def predict_mlp(model, features):
    model.eval()
    tensor = torch.as_tensor(features, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        return torch.sigmoid(model(tensor)).cpu().numpy()


def fit_mlp(X_fit, y_fit, X_valid=None, y_valid=None, epochs=200, patience=15):
    model = ChurnMLP(X_fit.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(
        TensorDataset(torch.as_tensor(X_fit, dtype=torch.float32), torch.as_tensor(y_fit, dtype=torch.float32)),
        batch_size=256, shuffle=True, generator=generator, pin_memory=True,
    )
    best_loss, best_state, best_epoch, stale = np.inf, None, 0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=True)
            batch_y = batch_y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
        if X_valid is None:
            continue
        model.eval()
        valid_x = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
        valid_y = torch.as_tensor(y_valid, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            valid_loss = criterion(model(valid_x), valid_y).item()
        if valid_loss < best_loss - 1e-5:
            best_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_epoch or epochs


def ordered_candidates(scores, customer_ids, threshold=BREAK_EVEN_PROBABILITY, top_k=None):
    scores = np.asarray(scores)
    customer_ids = np.asarray(customer_ids).astype(str)
    eligible = np.arange(len(scores)) if top_k is not None else np.flatnonzero(scores > threshold)
    ordered = eligible[np.lexsort((customer_ids[eligible], -scores[eligible]))]
    return ordered[:top_k] if top_k is not None else ordered


def simulate_budget(y_true, order, initial_budget=INITIAL_BUDGET, cost=INTERVENTION_COST, reward=SUCCESS_REWARD):
    budget = int(initial_budget)
    rows = []
    for rank, index in enumerate(order, start=1):
        if budget < cost:
            break
        before = budget
        budget -= cost
        success = int(y_true[index]) == 1
        if success:
            budget += reward
        rows.append((rank, int(index), before, success, budget, budget - initial_budget))
    ledger = pd.DataFrame(rows, columns=["rank", "row_index", "budget_before", "success", "budget_after", "cumulative_profit"])
    return {"final_budget": budget, "profit": budget - initial_budget, "intervened": len(ledger), "stopped_for_budget": len(ledger) < len(order)}, ledger


# Pruebas minimas del dinero y frontera de presupuesto.
assert simulate_budget(np.array([1]), [0], initial_budget=10)[0]["final_budget"] == 100
assert simulate_budget(np.array([0]), [0], initial_budget=10)[0]["final_budget"] == 0
assert simulate_budget(np.array([1]), [0], initial_budget=9)[0]["intervened"] == 0
assert simulate_budget(np.array([0, 1]), [0, 1], initial_budget=10)[0]["intervened"] == 1


def cpu_or_gpu_model(name):
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=2000, random_state=SEED), "CPU"
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=400, max_depth=10, min_samples_leaf=3, n_jobs=-1, random_state=SEED), "CPU"
    if name == "XGBoost":
        return XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.03, subsample=0.85, colsample_bytree=0.85, tree_method="hist", device="cuda", eval_metric="logloss", random_state=SEED), "CUDA T4"
    if name == "LightGBM":
        return LGBMClassifier(n_estimators=500, num_leaves=31, learning_rate=0.03, subsample=0.85, colsample_bytree=0.85, device_type="gpu", random_state=SEED, verbosity=-1), "GPU T4"
    raise KeyError(name)

## 4. Predicciones out-of-fold

Cinco folds estratificados producen scores de train sin prediccion sobre la misma fila usada para ajustar. Estos scores gobiernan calibracion y eleccion oficial.

In [ ]:
MODEL_NAMES = ["MLP", "CatBoost", "LightGBM", "Logistic Regression", "Random Forest", "XGBoost"]

def generate_oof_predictions():
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = {name: np.zeros(len(X_train), dtype=float) for name in MODEL_NAMES}
    times = {name: 0.0 for name in MODEL_NAMES}
    mlp_epochs = []

    for fold, (fit_idx, valid_idx) in enumerate(splitter.split(X_train, y_train), start=1):
        fold_preprocessor = make_preprocessor()
        X_fit_encoded = fold_preprocessor.fit_transform(X_train.iloc[fit_idx]).astype(np.float32)
        X_valid_encoded = fold_preprocessor.transform(X_train.iloc[valid_idx]).astype(np.float32)

        started = time.perf_counter()
        mlp, best_epoch = fit_mlp(X_fit_encoded, y_train[fit_idx], X_valid_encoded, y_train[valid_idx])
        oof["MLP"][valid_idx] = predict_mlp(mlp, X_valid_encoded)
        times["MLP"] += time.perf_counter() - started
        mlp_epochs.append(best_epoch)

        cat_fit = X_train.iloc[fit_idx].copy()
        cat_valid = X_train.iloc[valid_idx].copy()
        cat_fit[categorical_columns] = cat_fit[categorical_columns].astype(str)
        cat_valid[categorical_columns] = cat_valid[categorical_columns].astype(str)
        cat = CatBoostClassifier(
            iterations=500, depth=6, learning_rate=0.03, loss_function="Logloss",
            eval_metric="Logloss", task_type="GPU", devices="0", random_seed=SEED,
            verbose=False, allow_writing_files=False,
        )
        started = time.perf_counter()
        cat.fit(cat_fit, y_train[fit_idx], cat_features=categorical_columns, eval_set=(cat_valid, y_train[valid_idx]), early_stopping_rounds=50, verbose=False)
        oof["CatBoost"][valid_idx] = cat.predict_proba(cat_valid)[:, 1]
        times["CatBoost"] += time.perf_counter() - started

        for name in ["LightGBM", "Logistic Regression", "Random Forest", "XGBoost"]:
            model, _ = cpu_or_gpu_model(name)
            started = time.perf_counter()
            model.fit(X_fit_encoded, y_train[fit_idx])
            oof[name][valid_idx] = model.predict_proba(X_valid_encoded)[:, 1]
            times[name] += time.perf_counter() - started

        print(f"fold={fold}/5 mlp_best_epoch={best_epoch}")

    assert all(np.isfinite(values).all() for values in oof.values())
    return oof, times, mlp_epochs


run_summary["stage"] = "oof_training"
save_summary()
oof_raw, oof_training_times, mlp_best_epochs = generate_oof_predictions()
print({"median_mlp_epoch": int(np.median(mlp_best_epochs)), "models": MODEL_NAMES})

## 5. Calibracion y seleccion oficial

Platt e isotonic se comparan con validacion cruzada interna sobre scores OOF. El menor Brier gana; log-loss desempata. Luego el calibrador elegido se ajusta sobre todos los scores OOF.

In [ ]:
def fit_calibrator(method, scores, labels):
    if method == "platt":
        calibrator = LogisticRegression(random_state=SEED)
        calibrator.fit(np.asarray(scores).reshape(-1, 1), labels)
    else:
        calibrator = IsotonicRegression(out_of_bounds="clip")
        calibrator.fit(scores, labels)
    return calibrator


def apply_calibrator(calibrator, method, scores):
    scores = np.asarray(scores)
    if method == "platt":
        return calibrator.predict_proba(scores.reshape(-1, 1))[:, 1]
    return calibrator.predict(scores)


def choose_calibrator(scores, labels):
    split = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    candidates = []
    for method in ["platt", "isotonic"]:
        predictions = np.zeros(len(scores))
        for fit_idx, valid_idx in split.split(scores, labels):
            calibrator = fit_calibrator(method, scores[fit_idx], labels[fit_idx])
            predictions[valid_idx] = apply_calibrator(calibrator, method, scores[valid_idx])
        clipped = np.clip(predictions, 1e-6, 1 - 1e-6)
        candidates.append({"method": method, "brier": brier_score_loss(labels, clipped), "log_loss": log_loss(labels, clipped)})
    chosen = sorted(candidates, key=lambda row: (row["brier"], row["log_loss"]))[0]["method"]
    return chosen, fit_calibrator(chosen, scores, labels), candidates


calibrators = {}
calibration_rows = []
oof_calibrated = {}
validation_rows = []

for name in MODEL_NAMES:
    method, calibrator, candidate_rows = choose_calibrator(oof_raw[name], y_train)
    calibrators[name] = (method, calibrator)
    oof_calibrated[name] = apply_calibrator(calibrator, method, oof_raw[name])
    for row in candidate_rows:
        calibration_rows.append({"model": name, **row, "selected": row["method"] == method})
    order = ordered_candidates(oof_calibrated[name], ids_train)
    result, _ = simulate_budget(y_train, order)
    validation_rows.append({
        "model": name,
        "selection_source": "OOF train (valido)",
        "calibrator": method,
        "roc_auc_oof": roc_auc_score(y_train, oof_calibrated[name]),
        "pr_auc_oof": average_precision_score(y_train, oof_calibrated[name]),
        "brier_oof": brier_score_loss(y_train, oof_calibrated[name]),
        "profit_oof": result["profit"],
        "planned_interventions_oof": len(order),
        "executed_interventions_oof": result["intervened"],
    })

validation_metrics = pd.DataFrame(validation_rows).sort_values(["profit_oof", "pr_auc_oof"], ascending=False).reset_index(drop=True)
official_model = validation_metrics.loc[0, "model"]
assert official_model in MODEL_NAMES
pd.DataFrame(calibration_rows).to_csv(OUTPUT_DIR / "calibration_metrics.csv", index=False)
print(validation_metrics.round(4).to_string(index=False))
print(f"Modelo oficial elegido sin mirar test: {official_model}")

## 6. Ajuste final y apertura unica de test

Todos los modelos se ajustan sobre train completo para construir la tabla diagnostica. El ganador oficial ya quedo congelado en la seccion anterior.

In [ ]:
def fit_all_and_predict_test():
    final_preprocessor = make_preprocessor()
    X_train_encoded = final_preprocessor.fit_transform(X_train).astype(np.float32)
    X_test_encoded = final_preprocessor.transform(X_test).astype(np.float32)
    raw_predictions = {}
    fit_times = {}
    devices = {
        "MLP": "CUDA T4", "CatBoost": "GPU T4", "LightGBM": "GPU T4",
        "Logistic Regression": "CPU baseline", "Random Forest": "CPU baseline", "XGBoost": "CUDA T4",
    }

    epochs = int(np.median(mlp_best_epochs))
    started = time.perf_counter()
    mlp, _ = fit_mlp(X_train_encoded, y_train, epochs=epochs)
    raw_predictions["MLP"] = predict_mlp(mlp, X_test_encoded)
    fit_times["MLP"] = time.perf_counter() - started

    cat_train = X_train.copy()
    cat_test = X_test.copy()
    cat_train[categorical_columns] = cat_train[categorical_columns].astype(str)
    cat_test[categorical_columns] = cat_test[categorical_columns].astype(str)
    cat = CatBoostClassifier(
        iterations=500, depth=6, learning_rate=0.03, loss_function="Logloss",
        task_type="GPU", devices="0", random_seed=SEED, verbose=False, allow_writing_files=False,
    )
    started = time.perf_counter()
    cat.fit(cat_train, y_train, cat_features=categorical_columns, verbose=False)
    raw_predictions["CatBoost"] = cat.predict_proba(cat_test)[:, 1]
    fit_times["CatBoost"] = time.perf_counter() - started

    for name in ["LightGBM", "Logistic Regression", "Random Forest", "XGBoost"]:
        model, _ = cpu_or_gpu_model(name)
        started = time.perf_counter()
        model.fit(X_train_encoded, y_train)
        raw_predictions[name] = model.predict_proba(X_test_encoded)[:, 1]
        fit_times[name] = time.perf_counter() - started

    return raw_predictions, fit_times, devices, epochs


run_summary["stage"] = "final_training"
save_summary()
test_raw, final_training_times, model_devices, final_mlp_epochs = fit_all_and_predict_test()
test_calibrated = {
    name: apply_calibrator(calibrators[name][1], calibrators[name][0], test_raw[name])
    for name in MODEL_NAMES
}

## 7. Resultados de modelos y estrategia

La tabla de test es retrospectiva. No reemplaza la seleccion OOF. Para cada modelo se aplica el umbral economico calibrado de 0.10 y el simulador secuencial.

In [ ]:
test_rows = []
test_ledgers = {}
for name in MODEL_NAMES:
    order = ordered_candidates(test_calibrated[name], ids_test)
    result, ledger = simulate_budget(y_test, order)
    test_ledgers[name] = ledger
    test_rows.append({
        "model": name,
        "test_analysis": "retrospectivo, no valido para seleccion",
        "roc_auc_test": roc_auc_score(y_test, test_calibrated[name]),
        "pr_auc_test": average_precision_score(y_test, test_calibrated[name]),
        "brier_test": brier_score_loss(y_test, test_calibrated[name]),
        "profit_test": result["profit"],
        "planned_interventions_test": len(order),
        "executed_interventions_test": result["intervened"],
        "final_budget_test": result["final_budget"],
        "stopped_for_budget": result["stopped_for_budget"],
    })

test_metrics = pd.DataFrame(test_rows).sort_values(["profit_test", "pr_auc_test"], ascending=False).reset_index(drop=True)
retrospective_winner = test_metrics.loc[0, "model"]
model_metrics = validation_metrics.merge(test_metrics, on="model", how="left")
model_metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)

training_times = pd.DataFrame({
    "model": MODEL_NAMES,
    "device": [model_devices[name] for name in MODEL_NAMES],
    "oof_seconds": [oof_training_times[name] for name in MODEL_NAMES],
    "final_fit_seconds": [final_training_times[name] for name in MODEL_NAMES],
})
training_times.to_csv(OUTPUT_DIR / "training_times.csv", index=False)

official_scores = test_calibrated[official_model]
official_order = ordered_candidates(official_scores, ids_test)
official_result, official_ledger = simulate_budget(y_test, official_order)
official_ledger.insert(2, "customerID", ids_test[official_ledger["row_index"].to_numpy()])
official_ledger.insert(3, "calibrated_probability", official_scores[official_ledger["row_index"].to_numpy()])
official_ledger.insert(4, "actual_churn", y_test[official_ledger["row_index"].to_numpy()])
official_ledger.to_csv(OUTPUT_DIR / "intervention_ledger.csv", index=False)

top_k = int(round(len(y_test) * 0.20))
top20_order = ordered_candidates(official_scores, ids_test, top_k=top_k)
top20_result, _ = simulate_budget(y_test, top20_order)

rng = np.random.default_rng(SEED)
random_profits = []
for _ in range(1000):
    random_order = rng.permutation(len(y_test))[:len(official_order)]
    random_profits.append(simulate_budget(y_test, random_order)[0]["profit"])

oracle_order = np.flatnonzero(y_test == 1)
oracle_result, _ = simulate_budget(y_test, oracle_order)

strategy_comparison = pd.DataFrame([
    {"strategy": f"Modelo oficial {official_model}, p>0.10", "profit": official_result["profit"], "interventions": official_result["intervened"], "comparison_note": "seleccion valida por OOF"},
    {"strategy": f"{official_model} Top-20%", "profit": top20_result["profit"], "interventions": top20_result["intervened"], "comparison_note": "regla anterior con modelo nuevo"},
    {"strategy": "Aleatorio (media 1000 repeticiones)", "profit": float(np.mean(random_profits)), "interventions": len(official_order), "comparison_note": "misma cantidad planeada"},
    {"strategy": "Aleatorio percentil 95", "profit": float(np.percentile(random_profits, 95)), "interventions": len(official_order), "comparison_note": "misma cantidad planeada"},
    {"strategy": "Oraculo", "profit": oracle_result["profit"], "interventions": oracle_result["intervened"], "comparison_note": "techo teorico, usa etiquetas"},
    {"strategy": "Historico XGBoost Top-20%", "profit": 16080, "interventions": 282, "comparison_note": "no comparable: restriccion anterior"},
])
strategy_comparison.to_csv(OUTPUT_DIR / "strategy_comparison.csv", index=False)

assert official_result["final_budget"] - INITIAL_BUDGET == official_result["profit"]
if len(official_ledger):
    assert int(official_ledger.iloc[-1]["budget_after"]) == official_result["final_budget"]

print(model_metrics.round(4).to_string(index=False))
print(strategy_comparison.to_string(index=False))

## 8. Curvas y artefactos finales

Se guardan dos figuras legibles: evolucion del profit por intervencion y calibracion cruda frente a calibrada del modelo oficial.

In [ ]:
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(official_ledger["rank"], official_ledger["cumulative_profit"], color="#2563eb", linewidth=2)
ax.axhline(16080, color="#dc2626", linestyle="--", label="Historico S/16 080 (regla anterior)")
ax.set(title=f"Profit secuencial - modelo oficial {official_model}", xlabel="Intervenciones ejecutadas", ylabel="Profit acumulado (S/)")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "profit_curve.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
for label, values, color in [
    ("Sin calibrar", test_raw[official_model], "#f59e0b"),
    (f"Calibrada ({calibrators[official_model][0]})", official_scores, "#2563eb"),
]:
    observed, predicted = calibration_curve(y_test, values, n_bins=10, strategy="quantile")
    ax.plot(predicted, observed, marker="o", label=label, color=color)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Calibracion perfecta")
ax.set(title=f"Calibracion en test - {official_model}", xlabel="Probabilidad predicha", ylabel="Frecuencia observada")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "calibration_curve.png", dpi=160)
plt.show()

official_test_row = test_metrics.set_index("model").loc[official_model]
improved = bool(official_result["profit"] > 16080)
report = f"""# Segunda iteracion - resultados ejecutados

## Decision oficial

- Modelo: **{official_model}**, elegido por profit OOF sobre train.
- Calibracion: **{calibrators[official_model][0]}**.
- Profit secuencial test: **S/{official_result['profit']:,.0f}**.
- Presupuesto final: **S/{official_result['final_budget']:,.0f}**.
- Intervenciones ejecutadas: **{official_result['intervened']}** de {len(official_order)} planeadas.
- Detencion por falta de presupuesto: **{'si' if official_result['stopped_for_budget'] else 'no'}**.
- Supera S/16 080 historico: **{'si' if improved else 'no'}**; comparacion no equivalente porque cambio la restriccion.

## Calidad predictiva en test

- ROC-AUC: {official_test_row['roc_auc_test']:.4f}
- PR-AUC: {official_test_row['pr_auc_test']:.4f}
- Brier: {official_test_row['brier_test']:.4f}

## Diagnostico retrospectivo

Ganador mirando test: **{retrospective_winner}**. Este dato es **retrospectivo, no valido para seleccion** y no sustituye al modelo oficial.

## Metodo

Split 80/20 estratificado, semilla 42. Seleccion y calibracion mediante predicciones OOF de train. Umbral economico `p(churn) > 0.10`; ranking descendente con desempate por `customerID`. Cada intervencion resta S/10 y cada exito suma S/100.

## Caveats

- El resultado historico S/16 080 uso Top-20%; sirve como referencia, no como comparacion causal.
- Operaciones GPU pueden variar levemente pese a semilla fija.
- Elegir el ganador por test seria fuga; por eso solo se reporta retrospectivamente.
"""
(OUTPUT_DIR / "second_iteration.md").write_text(report, encoding="utf-8")

run_summary.update({
    "status": "complete",
    "stage": "done",
    "official_model": official_model,
    "selection_basis": "OOF train sequential profit",
    "retrospective_test_winner": retrospective_winner,
    "retrospective_warning": "retrospectivo, no valido para seleccion",
    "calibrator": calibrators[official_model][0],
    "mlp_final_epochs": final_mlp_epochs,
    "initial_budget": INITIAL_BUDGET,
    "final_budget": official_result["final_budget"],
    "profit": official_result["profit"],
    "planned_interventions": len(official_order),
    "executed_interventions": official_result["intervened"],
    "stopped_for_budget": official_result["stopped_for_budget"],
    "improved_vs_historical_16080": improved,
    "artifacts": [
        "intervention_ledger.csv", "model_metrics.csv", "strategy_comparison.csv",
        "calibration_metrics.csv", "training_times.csv", "profit_curve.png",
        "calibration_curve.png", "run_summary.json", "second_iteration.md",
    ],
})
save_summary()
print(json.dumps(run_summary, indent=2, ensure_ascii=False))

## 9. Takeaways

Usar `second_iteration.md` y `run_summary.json` como resumen ejecutado. El modelo oficial siempre corresponde a validacion OOF; cualquier ganador por test queda explicitamente marcado como retrospectivo.